# Actin Retrograde Flow Analyzer

Fully automated measurement of F-actin retrograde flow rates in neuronal growth cones from TIRF microscopy image stacks.

**Pipeline:**
1. Load nd2 or TIFF image stack
2. Auto-detect growth cone and place measurement lines
3. Generate kymographs along each line
4. Claude Sonnet reads diagonal streak slopes from kymographs
5. Compute flow rates via point-slope form

---

## 1. Setup

In [ ]:
!pip install -q numpy matplotlib scipy tifffile anthropic nd2 boxsdk

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, io, re, json, base64, glob
from pathlib import Path
from scipy.ndimage import (map_coordinates, label, binary_dilation,
                           binary_erosion, gaussian_filter, center_of_mass)
import tifffile
import anthropic

%matplotlib inline

## 2. API Key

Enter your Anthropic API key. You can get one at https://console.anthropic.com/.

In [ ]:
from getpass import getpass

# Option 1: Enter key interactively (recommended)
ANTHROPIC_API_KEY = getpass("Enter your Anthropic API key: ")

# Option 2: Uncomment and paste directly (less secure)
# ANTHROPIC_API_KEY = "sk-ant-..."

print(f"Key set: {ANTHROPIC_API_KEY[:12]}...")

## 3. Load Data

Choose **one** of the three options below: upload files directly, mount Google Drive, or pull from Box.

In [ ]:
# === OPTION A: Upload files directly ===
from google.colab import files
uploaded = files.upload()
DATA_FILES = list(uploaded.keys())
print(f"Uploaded {len(DATA_FILES)} file(s): {DATA_FILES}")

In [ ]:
# === OPTION B: Mount Google Drive ===
# Uncomment and run this cell instead of Option A if your files are on Drive.

# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/path/to/your/nd2/files'
# DATA_FILES = sorted(glob.glob(os.path.join(DATA_DIR, '*.nd2'))) + \
#              sorted(glob.glob(os.path.join(DATA_DIR, '*.tif')))
# print(f"Found {len(DATA_FILES)} file(s)")
# for f in DATA_FILES:
#     print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e6:.1f} MB")

In [ ]:
# === OPTION C: Download from Box ===
# Uses a Box Developer Token (valid 60 min) or JWT auth.
#
# To get a Developer Token:
#   1. Go to https://app.box.com/developers/console
#   2. Create a new app (or use existing) → Configuration tab
#   3. Under "Developer Token", click "Generate Developer Token"
#   4. Paste it below
#
# Set BOX_FOLDER_ID to the folder containing your nd2 files.
# You can find this in the URL when viewing the folder on box.com:
#   https://ucsf.app.box.com/folder/123456789  →  BOX_FOLDER_ID = '123456789'

BOX_DEVELOPER_TOKEN = ''  # paste your token here
BOX_FOLDER_ID = ''        # paste your folder ID here
BOX_DOWNLOAD_DIR = '/content/box_data'

if BOX_DEVELOPER_TOKEN and BOX_FOLDER_ID:
    from boxsdk import OAuth2, Client
    os.makedirs(BOX_DOWNLOAD_DIR, exist_ok=True)

    auth = OAuth2(
        client_id='',  # not needed for developer tokens
        client_secret='',
        access_token=BOX_DEVELOPER_TOKEN,
    )
    client = Client(auth)

    folder = client.folder(BOX_FOLDER_ID)
    items = folder.get_items()

    DATA_FILES = []
    for item in items:
        if item.name.lower().endswith(('.nd2', '.tif', '.tiff')):
            local_path = os.path.join(BOX_DOWNLOAD_DIR, item.name)
            if not os.path.exists(local_path):
                print(f"  Downloading: {item.name} ({item.size/1e6:.1f} MB)...",
                      end='', flush=True)
                with open(local_path, 'wb') as f:
                    client.file(item.id).download_to(f)
                print(" done")
            else:
                print(f"  Already downloaded: {item.name}")
            DATA_FILES.append(local_path)

    print(f"\n{len(DATA_FILES)} file(s) from Box:")
    for f in DATA_FILES:
        print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e6:.1f} MB")
else:
    print("Box not configured — fill in BOX_DEVELOPER_TOKEN and BOX_FOLDER_ID above.")

## 4. Pipeline Functions

In [ ]:
# ============================================================
# Loading
# ============================================================

def load_stack(path):
    """Load a TIRF image stack. Returns (stack, pixel_size, frame_interval)."""
    p = Path(path)
    pixel_size = None
    frame_interval = None

    if p.suffix.lower() == '.nd2':
        import nd2
        with nd2.ND2File(str(p)) as f:
            stack = f.asarray()
            pixel_size = f.voxel_size().x
            events = f.events()
            if events and len(events) >= 2:
                t0 = events[0].get('Time [s]', 0)
                t1 = events[1].get('Time [s]', 0)
                if t1 > t0:
                    frame_interval = t1 - t0
            if hasattr(f, 'experiment') and f.experiment:
                for loop in f.experiment:
                    if hasattr(loop, 'parameters') and hasattr(loop.parameters, 'periodMs'):
                        frame_interval = loop.parameters.periodMs / 1000.0
            print(f"nd2: {f.sizes}")
            if f.metadata and hasattr(f.metadata, 'channels') and f.metadata.channels:
                for ch in f.metadata.channels:
                    print(f"  Channel: {ch.channel.name}")
    elif p.is_dir():
        files_list = sorted(p.glob('*.tif')) + sorted(p.glob('*.tiff'))
        stack = np.stack([tifffile.imread(str(f)) for f in files_list])
    else:
        stack = tifffile.imread(str(p))

    if stack.ndim == 5:
        stack = stack[:, 0, 0]
    elif stack.ndim == 4:
        stack = stack[:, 0]

    pixel_size = pixel_size or 0.108
    frame_interval = frame_interval or 2.0

    print(f"Loaded: {stack.shape[0]} frames, {stack.shape[1]}x{stack.shape[2]} px, "
          f"dtype={stack.dtype}")
    print(f"Pixel size: {pixel_size} um/px, Frame interval: {frame_interval:.2f} s")
    return stack.astype(np.float64), pixel_size, frame_interval

In [ ]:
# ============================================================
# Growth cone detection + line placement
# ============================================================

def find_growth_cone(frame):
    """Detect growth cone: find neuron tip and flow direction."""
    smooth = gaussian_filter(frame, sigma=3)
    thresh = np.percentile(smooth[smooth > 0], 75)
    mask = smooth > thresh
    mask = binary_erosion(mask, iterations=2)
    mask = binary_dilation(mask, iterations=2)

    labeled, n = label(mask)
    if n == 0:
        return None
    sizes = [(labeled == i).sum() for i in range(1, n + 1)]
    neuron_mask = labeled == (np.argmax(sizes) + 1)

    cy, cx = center_of_mass(neuron_mask)
    ys, xs = np.where(neuron_mask)
    dists = np.hypot(xs - cx, ys - cy)
    tip_idx = np.argmax(dists)
    tip_x, tip_y = float(xs[tip_idx]), float(ys[tip_idx])

    gc_radius = min(frame.shape) * 0.15
    dist_from_tip = np.hypot(
        np.arange(frame.shape[1])[None, :] - tip_x,
        np.arange(frame.shape[0])[:, None] - tip_y)
    gc_mask = neuron_mask & (dist_from_tip < gc_radius)
    if gc_mask.sum() < 10:
        gc_mask = neuron_mask

    gc_ys, gc_xs = np.where(gc_mask)
    gc_cx, gc_cy = float(gc_xs.mean()), float(gc_ys.mean())

    flow_dx, flow_dy = cx - tip_x, cy - tip_y
    flow_len = np.hypot(flow_dx, flow_dy)
    if flow_len > 0:
        flow_dx /= flow_len
        flow_dy /= flow_len

    return dict(tip=(tip_x, tip_y), gc_center=(gc_cx, gc_cy),
                flow_direction=(float(flow_dx), float(flow_dy)))


def auto_place_lines(frame, n_lines=3, line_length_frac=0.35):
    """Auto-detect growth cone and place measurement lines."""
    gc = find_growth_cone(frame)
    if gc is None:
        print("Warning: Could not detect growth cone.")
        return []

    tip_x, tip_y = gc['tip']
    gc_cx, gc_cy = gc['gc_center']
    fdx, fdy = gc['flow_direction']
    H, W = frame.shape
    half = min(H, W) * line_length_frac / 2

    line_cx = tip_x + (gc_cx - tip_x) * 0.6
    line_cy = tip_y + (gc_cy - tip_y) * 0.6

    angles = [0]
    for i in range(1, (n_lines + 1) // 2 + 1):
        angles.extend([-30 * i, 30 * i])
    angles = angles[:n_lines]

    lines = []
    for ang_deg in angles:
        ang_rad = np.radians(ang_deg)
        dx = fdx * np.cos(ang_rad) - fdy * np.sin(ang_rad)
        dy = fdx * np.sin(ang_rad) + fdy * np.cos(ang_rad)
        x0 = np.clip(line_cx - half * dx, 1, W - 2)
        y0 = np.clip(line_cy - half * dy, 1, H - 2)
        x1 = np.clip(line_cx + half * dx, 1, W - 2)
        y1 = np.clip(line_cy + half * dy, 1, H - 2)
        lines.append(((x0, y0), (x1, y1)))

    print(f"  Growth cone: tip=({tip_x:.0f},{tip_y:.0f}), "
          f"flow=({fdx:.2f},{fdy:.2f}), {len(lines)} lines placed")
    return lines

In [ ]:
# ============================================================
# Kymograph generation
# ============================================================

def make_kymograph(stack, line, width=5):
    """Sample pixel intensities along `line` for every frame."""
    (x0, y0), (x1, y1) = line
    length = np.hypot(x1 - x0, y1 - y0)
    n_pts = int(np.ceil(length))
    dx, dy = (x1 - x0) / length, (y1 - y0) / length
    nx, ny = -dy, dx
    offsets = np.arange(-(width // 2), width // 2 + 1)
    t_vals = np.linspace(0, 1, n_pts)
    kymo = np.zeros((stack.shape[0], n_pts))
    for fi in range(stack.shape[0]):
        accum = np.zeros(n_pts)
        for off in offsets:
            xs = x0 + t_vals * (x1 - x0) + off * nx
            ys = y0 + t_vals * (y1 - y0) + off * ny
            accum += map_coordinates(stack[fi], np.vstack([ys, xs]),
                                    order=1, mode='nearest')
        kymo[fi] = accum / len(offsets)
    return kymo


def render_kymograph_png(kymo, pixel_size, frame_interval, label=''):
    """Render kymograph as PNG bytes with calibrated axes."""
    t_ext = kymo.shape[0] * frame_interval
    d_ext = kymo.shape[1] * pixel_size
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(kymo, cmap='gray', aspect='auto', extent=[0, d_ext, t_ext, 0])
    ax.set_xlabel('Distance (um)', fontsize=14)
    ax.set_ylabel('Time (s)', fontsize=14)
    if label:
        ax.set_title(label, fontsize=14)
    ax.tick_params(labelsize=12)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    return buf.read()

In [ ]:
# ============================================================
# Vision model flow rate measurement
# ============================================================

KYMOGRAPH_PROMPT = """You are analyzing a kymograph image from TIRF microscopy of actin retrograde flow in a neuronal growth cone.

In this image:
- The x-axis shows distance in micrometers (um).
- The y-axis shows time in seconds (s), increasing downward.
- Diagonal streaks represent moving actin features.
- The slope of a streak gives the flow velocity.

Your task:
1. Identify the clearest diagonal streak(s) — these are lines that are tilted, NOT vertical (vertical = stationary).
2. For each streak, read two points precisely from the axis labels: (x1 um, t1 s) and (x2 um, t2 s).
3. Compute the flow rate for each streak: rate = |x2 - x1| / |t2 - t1| * 60 um/min

You MUST respond in this exact JSON format and nothing else:
{"streaks": [{"x1": 0.0, "t1": 0.0, "x2": 0.0, "t2": 0.0, "rate_um_per_min": 0.0}]}"""


def measure_flow_rate(kymo_png, api_key, model='claude-sonnet-4-6'):
    """Send kymograph to Sonnet and extract flow rate."""
    client = anthropic.Anthropic(api_key=api_key)
    img_b64 = base64.b64encode(kymo_png).decode()

    message = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64',
                 'media_type': 'image/png', 'data': img_b64}},
                {'type': 'text', 'text': KYMOGRAPH_PROMPT},
            ],
        }],
    )

    text = message.content[0].text.strip()
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{[\s\S]*\}', text)
        if match:
            try:
                data = json.loads(match.group())
            except json.JSONDecodeError:
                return {'streaks': [], 'rate_um_per_min': 0.0}
        else:
            return {'streaks': [], 'rate_um_per_min': 0.0}

    valid = []
    for s in data.get('streaks', []):
        try:
            x1, t1 = float(s['x1']), float(s['t1'])
            x2, t2 = float(s['x2']), float(s['t2'])
            dt = abs(t2 - t1)
            dx = abs(x2 - x1)
            if dt > 0.1:
                rate = dx / dt * 60.0
                valid.append(dict(x1=x1, t1=t1, x2=x2, t2=t2,
                                  rate_um_per_min=rate))
        except (KeyError, ValueError, TypeError):
            continue

    if not valid:
        return {'streaks': [], 'rate_um_per_min': 0.0}
    return {'streaks': valid,
            'rate_um_per_min': float(np.mean([s['rate_um_per_min'] for s in valid]))}

## 5b. Run Analysis

Process all uploaded files. Each file takes ~30 seconds and costs ~$0.03.

In [ ]:
# Configuration
N_LINES = 3        # lines per growth cone
LINE_WIDTH = 5     # pixel averaging width
MODEL = 'claude-sonnet-4-6'

# Override pixel_size / frame_interval here if not using nd2
# (nd2 files auto-detect these from metadata)
PIXEL_SIZE_OVERRIDE = None    # e.g., 0.108
FRAME_INTERVAL_OVERRIDE = None  # e.g., 3.0

## 5a. Cost Estimate

Run this cell to see the estimated API cost **before** making any calls.

In [ ]:
all_results = []

for filepath in DATA_FILES:
    print(f"\n{'='*60}")
    print(f"Processing: {os.path.basename(filepath)}")
    print(f"{'='*60}")

    # Load
    stack, pixel_size, frame_interval = load_stack(filepath)
    if PIXEL_SIZE_OVERRIDE:
        pixel_size = PIXEL_SIZE_OVERRIDE
    if FRAME_INTERVAL_OVERRIDE:
        frame_interval = FRAME_INTERVAL_OVERRIDE

    # Detect growth cone and place lines
    lines = auto_place_lines(stack[0], n_lines=N_LINES)
    if not lines:
        print("  SKIPPED: could not detect growth cone")
        continue

    # Generate kymographs and measure
    kymographs = []
    measurements = []
    for i, line in enumerate(lines):
        kymo = make_kymograph(stack, line, width=LINE_WIDTH)
        kymographs.append(kymo)
        kymo_png = render_kymograph_png(kymo, pixel_size, frame_interval,
                                        f'L{i+1}')
        m = measure_flow_rate(kymo_png, ANTHROPIC_API_KEY, model=MODEL)
        measurements.append(m)
        n_streaks = len(m.get('streaks', []))
        print(f"  L{i+1}: {m['rate_um_per_min']:.1f} um/min "
              f"({n_streaks} streak(s))")

    # Store results
    rates = [m['rate_um_per_min'] for m in measurements]
    mean_rate = np.mean([r for r in rates if r > 0]) if any(r > 0 for r in rates) else 0.0
    all_results.append({
        'file': os.path.basename(filepath),
        'rates': rates,
        'mean_rate': mean_rate,
        'lines': lines,
        'kymographs': kymographs,
        'measurements': measurements,
        'stack_frame0': stack[0],
        'pixel_size': pixel_size,
        'frame_interval': frame_interval,
    })
    print(f"  Mean: {mean_rate:.1f} um/min")

print(f"\n\nProcessed {len(all_results)} file(s).")

## 6. Results

In [ ]:
# === Summary table ===
print(f"{'File':<40s} {'L1':>8s} {'L2':>8s} {'L3':>8s} {'Mean':>8s}")
print('-' * 75)
for r in all_results:
    rates_str = [f"{rate:>8.1f}" for rate in r['rates']]
    while len(rates_str) < 3:
        rates_str.append(f"{'N/A':>8s}")
    print(f"{r['file']:<40s} {''.join(rates_str[:3])} {r['mean_rate']:>8.1f}")

In [ ]:
# === Detailed figures for each file ===
# Layout: [Image with lines] [Rate table] [Kymograph L1] [Kymograph L2] [Kymograph L3]

from matplotlib.gridspec import GridSpec

for r in all_results:
    n = len(r['lines'])
    n_cols = n + 2
    fig = plt.figure(figsize=(4.5 * n_cols, 5.5))
    gs = GridSpec(1, n_cols, figure=fig, width_ratios=[1.2, 0.8] + [1.0] * n,
                  wspace=0.3)

    # --- Column 1: Growth cone image with lines ---
    ax_img = fig.add_subplot(gs[0, 0])
    frame0 = r['stack_frame0']
    ax_img.imshow(frame0, cmap='gray',
                  vmin=np.percentile(frame0, 1),
                  vmax=np.percentile(frame0, 99.5))
    colors = ['red', 'cyan', 'lime', 'yellow', 'magenta']
    for i, line in enumerate(r['lines']):
        (x0, y0), (x1, y1) = line
        ax_img.plot([x0, x1], [y0, y1], color=colors[i % len(colors)], lw=2)
        ax_img.text((x0+x1)/2, (y0+y1)/2, f'L{i+1}',
                    color=colors[i % len(colors)], fontsize=11, fontweight='bold')
    ax_img.set_title(r['file'], fontsize=10)
    ax_img.axis('off')

    # --- Column 2: Flow rate table ---
    ax_tbl = fig.add_subplot(gs[0, 1])
    ax_tbl.axis('off')

    table_data = []
    cell_colors = []
    for i, m in enumerate(r['measurements']):
        rate = m['rate_um_per_min']
        table_data.append([f'L{i+1}', f'{rate:.1f}'])
        cell_colors.append([colors[i % len(colors)], 'white'])

    mean_rate = r['mean_rate']
    table_data.append(['Mean', f'{mean_rate:.1f}'])
    cell_colors.append(['lightgray', 'lightgray'])

    tbl = ax_tbl.table(
        cellText=table_data,
        colLabels=['Line', 'Rate\n(um/min)'],
        cellColours=cell_colors,
        colColours=['#d9d9d9', '#d9d9d9'],
        cellLoc='center',
        loc='center',
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1.0, 1.8)

    for i in range(len(table_data)):
        cell = tbl[i + 1, 0]
        if i < len(r['measurements']):
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_text_props(fontweight='bold')
        rate_cell = tbl[i + 1, 1]
        if i == len(r['measurements']):
            rate_cell.set_text_props(fontweight='bold', fontsize=12)

    ax_tbl.set_title('Flow Rates', fontsize=11, fontweight='bold', pad=12)

    # --- Columns 3+: Kymographs ---
    ps = r['pixel_size']
    fi = r['frame_interval']
    for i, (kymo, m) in enumerate(zip(r['kymographs'], r['measurements'])):
        ax = fig.add_subplot(gs[0, i + 2])
        t_ext = kymo.shape[0] * fi
        d_ext = kymo.shape[1] * ps
        ax.imshow(kymo, cmap='gray', aspect='auto',
                  extent=[0, d_ext, t_ext, 0],
                  vmin=np.percentile(kymo, 2),
                  vmax=np.percentile(kymo, 98))
        for s in m.get('streaks', []):
            if 'x1' in s:
                ax.plot([s['x1'], s['x2']], [s['t1'], s['t2']],
                        'r-', lw=2, alpha=0.8)
                ax.plot(s['x1'], s['t1'], 'r+', ms=10, mew=2)
                ax.plot(s['x2'], s['t2'], 'r+', ms=10, mew=2)
        ax.set_xlabel('Distance (um)')
        if i == 0:
            ax.set_ylabel('Time (s)')
        else:
            ax.set_ylabel('')
            ax.tick_params(labelleft=False)
        ax.set_title(f'L{i+1}', color=colors[i % len(colors)],
                     fontweight='bold', fontsize=11)

    fig.tight_layout()
    plt.show()

In [ ]:
# === Bar chart summary ===
if all_results:
    fig, ax = plt.subplots(figsize=(max(8, len(all_results) * 1.5), 5))
    names = [r['file'].replace('.nd2', '').replace('.tif', '') for r in all_results]
    means = [r['mean_rate'] for r in all_results]
    bars = ax.bar(range(len(means)), means, color='steelblue', edgecolor='black')
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Actin flow rate (um/min)')
    ax.set_title('Actin Retrograde Flow')
    for i, v in enumerate(means):
        ax.text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=9)
    fig.tight_layout()
    plt.show()

## 7. Export

In [ ]:
# === Save CSV ===
import csv

csv_path = 'flow_rate_results.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['file', 'line', 'rate_um_per_min', 'n_streaks',
                'streak_x1', 'streak_t1', 'streak_x2', 'streak_t2'])
    for r in all_results:
        for i, m in enumerate(r['measurements']):
            streaks = m.get('streaks', [])
            if streaks:
                for s in streaks:
                    w.writerow([r['file'], f'L{i+1}',
                                f"{s['rate_um_per_min']:.2f}",
                                len(streaks),
                                f"{s.get('x1',0):.2f}",
                                f"{s.get('t1',0):.2f}",
                                f"{s.get('x2',0):.2f}",
                                f"{s.get('t2',0):.2f}"])
            else:
                w.writerow([r['file'], f'L{i+1}',
                            f"{m['rate_um_per_min']:.2f}", 0,
                            '', '', '', ''])

print(f"Saved: {csv_path}")

# Download in Colab
try:
    from google.colab import files
    files.download(csv_path)
except ImportError:
    pass

---

**Cost:** ~$0.03 per file (3 API calls at ~$0.01 each). A full dataset of 150 files costs approximately $4.

**Accuracy:** Claude Sonnet achieves MAE of 0.2 um/min on real 16-bit nd2 data, matching human expert readings.

**Source:** [github.com/agbien/actin-retrograde-flow](https://github.com/agbien/actin-retrograde-flow)

In [ ]:
# ============================================================
# Cost estimate — no API calls are made here
# ============================================================

# Sonnet 4.6 pricing (as of March 2026)
INPUT_COST_PER_MTOK = 3.00   # $/million input tokens
OUTPUT_COST_PER_MTOK = 15.00  # $/million output tokens

# Per API call: ~1600 tokens (image) + ~200 tokens (prompt) = ~1800 input
#               ~200 tokens output (JSON response)
TOKENS_IN_PER_CALL = 1800
TOKENS_OUT_PER_CALL = 200

n_files = len(DATA_FILES)
calls_per_file = N_LINES
total_calls = n_files * calls_per_file
total_input = total_calls * TOKENS_IN_PER_CALL
total_output = total_calls * TOKENS_OUT_PER_CALL
total_cost = (total_input * INPUT_COST_PER_MTOK + total_output * OUTPUT_COST_PER_MTOK) / 1e6

print("=" * 50)
print("  ESTIMATED API COST")
print("=" * 50)
print(f"  Files to process:      {n_files}")
print(f"  Lines per file:        {calls_per_file}")
print(f"  Total API calls:       {total_calls}")
print(f"  Input tokens:          ~{total_input:,}")
print(f"  Output tokens:         ~{total_output:,}")
print(f"  Model:                 {MODEL}")
print(f"  ---")
print(f"  Estimated cost:        ${total_cost:.2f}")
print("=" * 50)

if total_cost > 1.0:
    print(f"\n  ⚠ This will cost more than $1. Proceed?")
else:
    print(f"\n  ✓ Low cost — proceed to the next cell to run.")